# 01 — Exploratory Data Analysis

This notebook explores the AnomalyShield sample dataset: distributions, time series
patterns, rolling statistics, and data quality checks.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data.loader import TimeSeriesLoader

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load the raw CSV to keep the is_anomaly column for highlighting
df_raw = pd.read_csv("../assets/sample_data.csv", parse_dates=["date"])
df_raw = df_raw.set_index("date").sort_index()

# Also load via the official loader (returns only the value column with DatetimeIndex)
df = TimeSeriesLoader.from_csv("../assets/sample_data.csv")

print(f"Loaded {len(df)} rows from sample_data.csv")
df.head()

In [ ]:
# Basic statistics
print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print()
display(df.describe())
print()
df.info()

In [ ]:
# Time series plot with anomalies highlighted in red
anomaly_mask = df_raw["is_anomaly"] == 1

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_raw.index, df_raw["value"], color="steelblue", linewidth=0.8, label="Value")
ax.scatter(
    df_raw.index[anomaly_mask],
    df_raw["value"][anomaly_mask],
    color="red",
    s=40,
    zorder=5,
    label=f"Anomaly (n={anomaly_mask.sum()})",
)
ax.set_title("Time Series with Ground-Truth Anomalies")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of values
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["value"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Histogram of Values")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Frequency")

axes[1].boxplot(df["value"], vert=True)
axes[1].set_title("Box Plot of Values")
axes[1].set_ylabel("Value")

plt.tight_layout()
plt.show()

In [ ]:
# Rolling statistics (window=30)
window = 30
rolling_mean = df["value"].rolling(window=window).mean()
rolling_std = df["value"].rolling(window=window).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["value"], color="steelblue", alpha=0.4, linewidth=0.8, label="Raw Value")
ax.plot(df.index, rolling_mean, color="darkorange", linewidth=1.5, label=f"Rolling Mean ({window}d)")
ax.fill_between(
    df.index,
    rolling_mean - 2 * rolling_std,
    rolling_mean + 2 * rolling_std,
    alpha=0.15,
    color="darkorange",
    label=f"Rolling Mean +/- 2 Std",
)
ax.set_title("Rolling Statistics (window=30)")
ax.set_xlabel("Date")
ax.set_ylabel("Value")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Missing values check
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing: {missing.sum()} out of {len(df)} rows ({100 * missing.sum() / len(df):.2f}%)")

# Anomaly class balance
print("\nAnomaly distribution (ground truth):")
print(df_raw["is_anomaly"].value_counts())
print(f"Anomaly ratio: {df_raw['is_anomaly'].mean():.4f}")

## Summary

Key observations from this EDA:

1. **Data shape**: The sample dataset contains daily time series data with a `value` column and ground-truth `is_anomaly` labels.
2. **Anomalies**: A small fraction of points are marked as anomalies (spikes well above or below the normal range).
3. **Seasonality**: The time series shows visible weekly and annual seasonal patterns.
4. **Trend**: A slight upward linear trend is present.
5. **Missing values**: The dataset has no missing values -- ready for modeling.
6. **Stationarity**: The rolling mean shows a slow trend and the rolling standard deviation is relatively stable. Differencing or detrending may improve some detectors, but is not strictly required for the models used in AnomalyShield.

Next step: **02_detection_methods.ipynb** -- train and compare anomaly detection models.